# answer_truncation on BANKING77: damage, then fix against control

For Kaggle: Settings > GPU T4, Internet on; then Save Version > Save & Run All. About 15 training runs of Qwen3-0.6B, roughly 4 hours. See docs/answer-cut.md. Set `REVISION` to the commit to run.

In [ ]:
REVISION = "main"
!git clone -q https://github.com/MaxKharlan/next-experiment.git
%cd next-experiment
!git checkout -q {REVISION}
!git log --oneline -1

In [ ]:
!pip install -q "transformers==5.20.0" "peft==0.21.2" "pydantic>=2.14.1" pyyaml httpx
# Kaggle ships torchao 0.10, which peft 0.21 refuses; LoRA here does not need it
!pip uninstall -y -q torchao
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
!python -m bench.tasks.banking77 --out artifacts/tasks/banking77
!python -m bench.faults.data --task artifacts/tasks/banking77 --out artifacts/tasks/banking77-answers-cut-10 --strength 0.1
!python -m bench.faults.data --task artifacts/tasks/banking77 --out artifacts/tasks/banking77-answers-cut-30 --strength 0.3
!sha256sum artifacts/tasks/banking77/check.jsonl

In [ ]:
!PYTHONPATH=. python -m jobs.kaggle.answer_cut_effect
!cp runs/results.json /kaggle/working/a5_results.json

In [ ]:
!PYTHONPATH=. python -m jobs.kaggle.answer_cut_pair --strength 30
!cp runs/pair.json /kaggle/working/a7_pair.json